In [1]:
'''
***************************************************
Split train
***************************************************
'''

data = []
with open('./cleaned_data/big_matrix.txt', 'r') as f:
    for line in f:
        row = [int(x) for x in line.strip().split()]
        data.append(row)
print(f"Number of users for training: {len(data)}")

user_list = []
with open('./cleaned_data/small_matrix.txt', 'r') as f:
    for line in f:
        row = [int(x) for x in line.strip().split()]
        user_list.append(row[0])
print(f"Number of selected users for testing: {len(user_list)}")

import random
import math

def train_test_split_per_user(data, test_user, test_ratio=0.2, dropout=0.0, seed=42):
    random.seed(seed)
    train_data, test_data = [], []
    train_size = 0
    test_size = 0
    
    for u_items in data:
        items = u_items[1:]  # Exclude user ID
        
        if dropout > 0.0:
            keep_count = max(1, math.floor(len(items) * (1 - dropout))) 
            items = random.sample(items, keep_count)
        
        n_items = len(items)
        n_test = math.ceil(n_items * test_ratio)
        test_items = random.sample(items, n_test)
        train_items = [x for x in items if x not in test_items]
        train_size += len(train_items)
        train_data.append(u_items[:1] + train_items)
        if u_items[0] in test_user:
            test_size += len(test_items)
            test_data.append(u_items[:1] + test_items)

    print(f"Train size: {train_size}; Test size: {test_size}.")
    print(f"Train users: {len(train_data)}; Test users: {len(test_data)}.")
    return train_data, test_data

train_data, test_data = train_test_split_per_user(data, user_list, test_ratio=0.3, dropout=0.7, seed=42)
with open('./cleaned_data/train.txt', 'w', encoding='utf-8') as f:
    for row in train_data:
        f.write(' '.join(map(str, row)) + '\n')

with open('./cleaned_data/val.txt', 'w', encoding='utf-8') as f:
    for row in test_data:
        f.write(' '.join(map(str, row)) + '\n')

Number of users for training: 4294
Number of selected users for testing: 1411
Train size: 1015272; Test size: 39241.
Train users: 4294; Test users: 1411.


In [2]:
l = 10000
for row in train_data:
    l = min(l, len(row)-1)
print(f"Minimum train items per user: {l}")


l = 10000
for row in test_data:
    l = min(l, len(row)-1)
print(f"Minimum val items per user: {l}")



Minimum train items per user: 12
Minimum val items per user: 12


In [3]:
'''
***************************************************
Split test
Popularity policy
***************************************************
'''

# import numpy as np
# import pandas as pd
# data = pd.read_csv('./cleaned_data/small_matrix.csv')
# test_data = []
# with open('./cleaned_data/test.txt', 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         row = list(map(int, parts))
#         test_data.append(row)


# def split_pop_niche(ratio=0.2, save=False):
#     item_counts = data['vid'].value_counts()
#     item_num = len(item_counts)
#     popular_threshold = int(ratio * item_num)

#     popular_items = set(item_counts.index[:popular_threshold])
#     print(f"Number of top {ratio*100}% popular items: {len(popular_items)}")
    
#     test_head = []  
#     test_tail = [] 

#     for parts in test_data:
#         user = parts[:1]
#         items = parts[1:]
        
#         head_items = [item for item in items if item in popular_items]
#         tail_items = [item for item in items if item not in popular_items]
        
#         if head_items:
#             test_head.append(user + head_items)

#         if tail_items:
#             test_tail.append(user + tail_items)
    
#     print(f"Popular test min item number: {np.min([len(i)-1 for i in test_head])}") 
#     print(f"Popular test total item number: {np.sum([len(i)-1 for i in test_head])}") 
#     print(f"Niche test min item number: {np.min([len(i)-1 for i in test_tail])}") 
#     print(f"Niche test total item number: {np.sum([len(i)-1 for i in test_tail])}\n") 
    
#     if save:
#         with open(f'./cleaned_data/test_top_{ratio}.txt', 'w', encoding='utf-8') as f:
#             for row in test_head:
#                 f.write(' '.join(map(str, row)) + '\n')
        
#         with open(f'./cleaned_data/test_tail_{1-ratio}.txt', 'w', encoding='utf-8') as f:
#             for row in test_tail:
#                 f.write(' '.join(map(str, row)) + '\n')
                
# for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
#     split_pop_niche(ratio, save=True)


'\n***************************************************\nSplit test\nPopularity policy\n***************************************************\n'

In [4]:
'''
***************************************************
Split test
Randomized item drop policy
***************************************************
'''

# import random
# import numpy as np
# import pandas as pd
# data = pd.read_csv('./cleaned_data/small_matrix.csv')
# test_data = []
# with open('./cleaned_data/test.txt', 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         row = list(map(int, parts))
#         test_data.append(row)

# def split_random(drop_num, save=False, seed=42):
#     random.seed(seed)

#     all_items = list(data['vid'].unique())

#     remove_items = set(random.sample(all_items, drop_num))

#     new_test_data = []
#     removed_count = 0

#     for row in test_data:
#         user_id = row[0]
#         items = row[1:]
        
#         new_items = [i for i in items if i not in remove_items]
#         removed_count += len(items) - len(new_items)

#         new_test_data.append([user_id] + new_items)

#     print(f"Randomly removed {len(remove_items)} items，in total {removed_count} interactions.")
#     print(f"Minimum user degree: {np.min([len(i)-1 for i in new_test_data])}.\n") 
#     if save:
#         with open(f'./cleaned_data/test_filtered_{drop_num}.txt', 'w', encoding='utf-8') as f:
#             for row in new_test_data:
#                 f.write(' '.join(map(str, row)) + '\n')
                
# for num in [1500, 2000]:
#     split_random(drop_num=num, save=False, seed=42)

'\n***************************************************\nSplit test\nRandomized item drop policy\n***************************************************\n'

In [5]:
'''
***************************************************
Split test
Randomized interaction drop policy
***************************************************
'''

# import random
# import numpy as np
# import pandas as pd
# data = pd.read_csv('./cleaned_data/small_matrix.csv')
# test_data = []
# with open('./cleaned_data/test.txt', 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         row = list(map(int, parts))
#         test_data.append(row)

# def drop_interactions(ratio, save=False, seed=42):

#     random.seed(seed)
#     drop_count = int(ratio * 65375)
    
#     single_part = [] 
#     remain_part = []  
#     for row in test_data:
#         user, items = row[0], row[1:]
    
#         first_item = random.choice(items)
#         single_part.append([user, first_item])
#         remain_items = [x for x in items if x != first_item]
#         remain_part.append([user] + remain_items)

#     interaction_pool = []
#     for u_idx, row in enumerate(remain_part):
#         user, items = row[0], row[1:]
#         for it in items:
#             interaction_pool.append((u_idx, it))
#     total_interactions = len(interaction_pool)
#     print(f"Total number of interactions {total_interactions}.")

#     drop_n = min(drop_count, total_interactions)
#     drop_samples = set(random.sample(interaction_pool, drop_n))
#     print(f"Globally deleted {drop_n} interactions.")

#     new_remain = []
#     for u_idx, row in enumerate(remain_part):
#         user, items = row[0], row[1:]
#         kept_items = [it for it in items if (u_idx, it) not in drop_samples]
#         new_remain.append([user] + kept_items)

#     user_items_map = {}
#     for row in single_part:
#         user_items_map[row[0]] = set(row[1:])  
#     for row in new_remain:
#         user = row[0]
#         if user in user_items_map:
#             user_items_map[user].update(row[1:])
#         else:
#             user_items_map[user] = set(row[1:])

#     new_test_data = [[u] + list(v) for u, v in user_items_map.items()]
#     total_final_items = sum(len(r) - 1 for r in new_test_data)

#     print(f"Number of interactions reserved: {total_final_items}.")
#     print(f"Minimum user degree: {np.min([len(i)-1 for i in new_test_data])}.\n") 

#     if save:
#         out_path = f"./cleaned_data/test_drop_{ratio}.txt"
#         with open(out_path, "w", encoding="utf-8") as f:
#             for row in new_test_data:
#                 f.write(" ".join(map(str, row)) + "\n")

# for ratio in [0.5, 0.6, 0.7, 0.8, 0.9]:
#     drop_interactions(ratio, save=True)

'\n***************************************************\nSplit test\nRandomized interaction drop policy\n***************************************************\n'

In [6]:
'''
***************************************************
Split test
Item exposure probability policy
***************************************************
'''

# import random
# import numpy as np
# import pandas as pd
# data = pd.read_csv('./cleaned_data/small_matrix.csv')
# test_data = []
# with open('./cleaned_data/test.txt', 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         row = list(map(int, parts))
#         test_data.append(row)
        
# def p(alpha, beta, seed=42):
#     """
#     Generate item drop probabilities from a Beta distribution.
#     """
#     random.seed(seed)
#     all_items = list(data['vid'].unique())

#     probs = np.random.beta(alpha, beta, size=len(all_items))
#     p_dict = {item: float(p) for item, p in zip(all_items, probs)}
    
#     return p_dict

# def probabilistic_drop(alpha, beta, save=False, seed=42):

#     random.seed(seed)

#     p_dict = p(alpha, beta, seed)
    
#     dropped_count = 0
#     total_interactions = 0
#     new_test_data = []

#     for row in test_data:
#         user, items = row[0], row[1:]
#         kept_items = []
#         for it in items:
#             total_interactions += 1
#             prob = p_dict.get(it, 0) 
#             if random.random() >= prob: 
#                 kept_items.append(it)
#             else:
#                 dropped_count += 1
#         if len(kept_items) == 0:
#             first_item = random.choice(items)
#             kept_items.append(first_item)
#             dropped_count -= 1
#         new_test_data.append([user] + kept_items)

#     drop_ratio = dropped_count / total_interactions if total_interactions > 0 else 0
#     print(f"Dropped {dropped_count}/{total_interactions} interactions ({drop_ratio:.2%}).")
#     print(f"Minimum user degree: {np.min([len(i)-1 for i in new_test_data])}.\n") 
    
#     if save:
#         out_path = f"./cleaned_data/test_beta_{alpha}_{beta}.txt"
#         with open(out_path, "w", encoding="utf-8") as f:
#             for row in new_test_data:
#                 f.write(" ".join(map(str, row)) + "\n")

# for val in [1, 0.1, 0.01, 0.001]:
#     probabilistic_drop(alpha=val, beta=val, save=True, seed=42)

'\n***************************************************\nSplit test\nItem exposure probability policy\n***************************************************\n'